In [ ]:
n_iterations = 5

In [ ]:
import model
import importlib
importlib.reload(model)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mesa.visualization import SolaraViz, SpaceRenderer, make_plot_component
from mesa import batch_run
import numpy as np
import sys
import gc

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.options.display.float_format = "{:.3f}".format

init_variables = {
    # Variables

    "V": 0,
    "M_A": 0,
    # TODO: M_A at t=1 is actually not defined. but setting 0 is justified?

    # Some variables will be calculated in the model
    # or received from the environment
    "r": None,
    "rpe": None,

    # Parameters

    # Parameter for the exponential moving average of anger
    "lambda_A": np.linspace(0, 1, 20),

    # Controllability
    "C": np.linspace(0, 1, 20),

    # Learning rate
    "eta": np.linspace(0, 1, 20),

    # Discounting factor
    "gamma": np.linspace(0, 1, 20),
}

# We batch-run all parameter combinations.
# Because the model is stochastic, we run each combination multiple times
# (e.g., five iterations) to estimate the expected behavior.
# Seeding works as follows: each iteration is assigned a unique seed,
# and that same seed is used for all parameter combinations within that iteration.
# This ensures reproducibility while allowing multiple stochastic replications
# per parameter set.
rng = np.random.default_rng(12345)
seed_values = rng.integers(0, sys.maxsize, size=(n_iterations,))

for iteration, seed_value in enumerate(seed_values):
    print(f"Starting iteration {iteration}...", end="", flush=True)

    results = batch_run(
        model.IrritabilityModel,
        parameters=init_variables,
        data_collection_period=1,  # collect data for every step
        number_processes=16,
        rng=seed_value,
        max_steps=200,
    )

    results_df = pd.DataFrame(results)
    results_df["iteration"] = iteration

    results_df.to_parquet(f"output/002_fnr_value_learning_{iteration}.parquet")

    del results
    del results_df
    gc.collect()

    print(f"Starting iteration {iteration}... done.")

# Merge the iterations

In [ ]:
import pandas as pd

dfs = [
    pd.read_parquet(f"output/002_fnr_value_learning_{i}.parquet")
    for i in range(n_iterations)
]

merged = pd.concat(dfs, ignore_index=True)
merged.to_parquet("output/002_fnr_value_learning_all.parquet")


## Summary (expectation and standard errors)

In [ ]:
import pandas as pd

merged = pd.read_parquet("output/002_fnr_value_learning_all.parquet")

In [ ]:
import pandas as pd
import numpy as np
import gc

group_cols = ["Step", "lambda_A", "C", "eta", "gamma", "AgentID"]

grouped = merged.groupby(group_cols)

# Quick check that that we have the right number of groups
assert grouped.ngroups == 20 * 20 * 20 * 20 * 200

stats = grouped.agg(
    V_mean=("V", "mean"),
    V_std=("V", "std"),
    V_n=("V", "count"),
    M_A_mean=("M_A", "mean"),
    M_A_std=("M_A", "std"),
    M_A_n=("M_A", "count"),
).reset_index()

print("Computing standard errors...")

stats["V_stderr"] = stats["V_std"] / np.sqrt(stats["V_n"])
stats["M_A_stderr"] = stats["M_A_std"] / np.sqrt(stats["M_A_n"])

# summary = stats.drop(columns=["V_std", "V_n", "M_A_std", "M_A_n"])
summary = stats

print("Saving summary data frame...")

summary.to_parquet("output/002_fnr_value_learning_summary.parquet")

# del stats, summary
# gc.collect()

In [ ]:
pd.set_option('display.max_rows', None)
summary.head()

# Downcast data for dashboard

In [2]:
import pandas as pd
import gc

# Summary dataframe

summary = pd.read_parquet("output/002_fnr_value_learning_summary.parquet")

summary.info(memory_usage="deep")

unique_n = summary["V_n"].unique()
assert len(unique_n) == 1, "V_n is not constant!"
n_iterations = unique_n[0]

summary.drop(columns=["AgentID", "V_n", "M_A_n"], inplace=True)

for c in summary.select_dtypes(include="integer"):
    summary[c] = pd.to_numeric(summary[c], downcast="integer")

for c in summary.select_dtypes(include="float"):
    summary[c] = pd.to_numeric(summary[c], downcast="float")

summary.info(memory_usage="deep")

summary.to_parquet("output/dashboard/002_fnr_value_learning_summary.parquet")

# Write meta-info
lambda_A_vals = sorted(summary["lambda_A"].unique())
eta_vals      = sorted(summary["eta"].unique())
gamma_vals    = sorted(summary["gamma"].unique())
C_vals        = sorted(summary["C"].unique())

meta_df = pd.DataFrame({
    "n_iterations": [n_iterations],
    "lambda_A_vals": [lambda_A_vals],
    "eta_vals": [eta_vals],
    "gamma_vals": [gamma_vals],
    "C_vals": [C_vals]
})

# Write to parquet
meta_df.to_parquet("output/dashboard/meta_info.parquet",
                   engine="pyarrow")

del summary
gc.collect()

# Individual iterations

for i in range(n_iterations):
    print("-------------------------------------------------------------------")

    df = pd.read_parquet(f"output/002_fnr_value_learning_{i}.parquet")

    df.info(memory_usage="deep")

    df.drop(columns=["AgentID", "RunId", "iteration", "r", "rpe", "seed"],
            inplace=True)

    for c in df.select_dtypes(include="integer"):
        df[c] = pd.to_numeric(df[c], downcast="integer")

    for c in df.select_dtypes(include="float"):
        df[c] = pd.to_numeric(df[c], downcast="float")

    df.info(memory_usage="deep")

    df.to_parquet(f"output/dashboard/002_fnr_value_learning_{i}.parquet")

    del df
    gc.collect()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000000 entries, 0 to 31999999
Data columns (total 14 columns):
 #   Column      Dtype  
---  ------      -----  
 0   Step        int64  
 1   lambda_A    float64
 2   C           float64
 3   eta         float64
 4   gamma       float64
 5   AgentID     float64
 6   V_mean      float64
 7   V_std       float64
 8   V_n         int64  
 9   M_A_mean    float64
 10  M_A_std     float64
 11  M_A_n       int64  
 12  V_stderr    float64
 13  M_A_stderr  float64
dtypes: float64(11), int64(3)
memory usage: 3.3 GB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000000 entries, 0 to 31999999
Data columns (total 11 columns):
 #   Column      Dtype  
---  ------      -----  
 0   Step        int16  
 1   lambda_A    float32
 2   C           float32
 3   eta         float32
 4   gamma       float32
 5   V_mean      float32
 6   V_std       float32
 7   M_A_mean    float32
 8   M_A_std     float32
 9   V_stderr    float32
 10  M_A_stderr  fl

# A glimpse at the data

In [ ]:
print(results_df["C"].unique())

# results1 = results_df[results_df["RunId"] == 230]

# gamma = 0.57894737
# eta = 0.05263158
# lambda_A = 0.15789474
# C = 1

results1 = results_df[
    np.isclose(results_df["gamma"], 0.57894737) &
    np.isclose(results_df["eta"], 0.05263158) &
    np.isclose(results_df["lambda_A"], 0.84210526) &
    np.isclose(results_df["C"], 1)
]

# print(results_df.loc[22826, "eta"], repr(results_df.loc[22826, "eta"]))
# print(results_df.loc[22826, "lambda_A"], repr(results_df.loc[22826, "lambda_A"]))
# print(results_df.loc[22826, "C"], repr(results_df.loc[22826, "C"]))

# print(results_df["gamma"].dtype)


# results1.head(n=200)

# Prior to the first step, many things are not defined (e.g. action)
# results_df = results_df[results_df["Step"] > 0]

# results_df.head(n=200)

# results_df.columns

# Some plots

In [ ]:
g = sns.lineplot(data=results1,
                 y="V",
                 x="Step",
                 label="Value of state")

g = sns.lineplot(data=results1,
                 y=results1["M_A"],
                 x="Step",
                 color="Orange",
                 label="Anger/frustration")

# g.set(title="Anger and aggressive action",
#       ylabel="Value");
plt.axhline(y=0, color='black', linewidth=1)
plt.axvline(x=100, color='black', linewidth=1, linestyle="--")

# plt.xlim(0, 130)
# plt.ylim(0, 1)
plt.show()